# Idea 3 — Run the same audit on every Claude product, and publish the result

This notebook is the supporting analysis for **Idea 3** in [`PROPOSAL.md`](./PROPOSAL.md#idea-3--run-the-same-audit-on-every-claude-product-and-publish-the-result).

The thesis: the 288 prompts analyzed in this repo are one slice of one product. Anthropic ships system prompts in many places — claude.ai, the API, Projects, Skills, agent products. The welfare claim that the corpus trains compliance over reasoning is much stronger if it generalizes across every Anthropic prompt corpus, and much weaker if Claude Code is an outlier. From outside Anthropic, this can't be told.

The proposal is for Anthropic to run the same analyzer pipeline against each of its other system-prompt corpora and publish a comparison. The analyzer is rule-based, runs in <5 minutes on a laptop, and the corpus directory is configurable — swapping in any other prompt corpus is a one-line change.

Sections:

1. **Methodology summary** — the five metrics the audit publishes per corpus, with one-line definitions.
2. **Lexicon transparency** — pointers to where the rule-based classifiers live, and the rationale for the hand-curated approach.
3. **Mock comparison table** — what the published output should look like. The Claude Code row is filled from the current YAML; rows for claude.ai / API / Projects / Skills are placeholders Anthropic would fill in.
4. **Reproducibility note** — the single-line change required to point the producer at a different corpus.
5. **Conclusions / Recommendations / Limitations** *(opinion, not data)* — the slice of the executive-summary commentary specific to cross-product comparison.

In [ ]:
"""Setup: load YAML data — used to fill the Claude Code row of the mock comparison table."""
import importlib
import altair as alt
import pandas as pd

import prompt_analysis
importlib.reload(prompt_analysis)
from prompt_analysis import load_yaml, build_alt_df

alt.data_transformers.disable_max_rows()

data              = load_yaml()
alt_df            = build_alt_df(data)
corpus_block      = data["corpus"]
per_file_records  = data["files"]

n_files = len(per_file_records)
n_sents = corpus_block["n_sents"]
print(f"loaded {n_files} files / {n_sents:,} sentences from the Claude Code corpus")
print(f"this is one product — the proposal is to run the same pipeline on every other Claude product corpus")


## 1. Methodology — five metrics per corpus

The five metrics the audit computes per corpus, in the order the proposal lists them:

1. **Rule-explanation share** (`pct_explained_para`) — of all rule-bearing sentences (imperative marker / hard prohibition / grammatically imperative), what fraction sit in a paragraph that contains any justification keyword (`because`, `due to`, `in order to`, `so that`, `to ensure`, `otherwise`, `since`, …). The Tier-1 headline metric. Producer: cell that builds `metrics.rule_explanation` in `00_data_pipeline.ipynb`.

2. **Judgment-vs-procedural cue ratio** (`judgment_to_procedural_ratio`) — count of words inviting model judgment (`decide`, `consider`, `evaluate`, `weigh`, …) divided by count of words prescribing procedure (`if X, then …`, `whenever …`, `step 1 …`). Currently 0.140 in Claude Code; procedural cues are 7× more common.

3. **Threat-vs-causal share among existing explanations** (`threat_share`) — of paragraphs that *do* explain a rule, what fraction use threat-style framing (`will fail`, `or else`, `is forbidden`) instead of causal-style (`because`, `due to`, `that's why`). Currently 0.448 in Claude Code.

4. **Interpersonal / gratitude register counts** — sentence counts of the `appreciative` and `collaborative` pragmatic-register classes. Currently 4 and 29 in Claude Code respectively, both near zero. The full 6-class `sentence_register` block in the YAML carries the rest.

5. **Cumulative trend of (1) over the product's release history** — at each release version V, the running mean of `pct_explained_para` across every prompt with `version ≤ V`. The shape of this curve over time is the per-release accountability signal.

The audit publishes one row per corpus with these five numbers (and per-category breakdowns where the corpus is large enough). Five numbers per corpus is enough to see whether what looks like a Claude Code pattern is product-specific or pattern-wide.

## 2. Lexicon transparency

Every metric above traces to a hand-curated lexicon plus (in some cases) a small parse-tree heuristic on top of spaCy's English model. No embeddings, no model judgments, no proprietary tooling. The full lexicon set is echoed verbatim into the YAML output under `lexicons`:

- `JUSTIFICATION_PATTERNS` — the keyword set for "explained" detection.
- `JUDGMENT_VERBS` and `PROCEDURAL_CUES` — the two halves of the judgment-vs-procedural ratio.
- `THREAT_PATTERNS` and `CAUSAL_PATTERNS` — the two halves of the threat_share metric.
- `IMPERATIVE_MARKERS`, `VOCAB.hard_prohibitions`, `SENTENCE_REGISTER_MARKERS`, `STANCE_MARKERS`, `REGISTER_MARKERS` — the rule-detection and pragmatic-register lexicons.
- `APOLOGY_MARKERS`, `ADDRESS_FORM_PATTERNS` — the structural-absence metrics.
- `RULES_HEADING_RE` — the regex for the (mostly absent) RULES section heading.

Rationale for hand-curated lexicons over external sentiment libraries (VADER, Hu-Liu, spacytextblob): every metric is **audit-traceable**. A reader can look at the keyword list and the per-sentence parquet (`sentences_classified.parquet`) and reconstruct exactly why a sentence was flagged. Sentiment libraries trade audit-traceability for breadth of coverage; for a welfare submission, audit-traceability is the right tradeoff.

Per-language-or-domain extension: the lexicons are Python lists in `00_data_pipeline.ipynb`. Adding terms (e.g. for a different product's vocabulary) is a list-append. Translating to another language is a list-replace. Anthropic can extend or localize without touching the analyzer logic.

## 3. Mock comparison table — what the published output should look like

One row per corpus, the five metrics as columns. Only the **Claude Code** row is filled (computed live from this repo's YAML); the other rows are placeholders that Anthropic would fill from their internal corpora.

A reader looking at this table after Anthropic publishes it would be able to answer the welfare claim's central question in one glance: is the rule-without-reason pattern (24% explained, 0.14 judgment ratio, 45% threat-share, ~0 gratitude register) a Claude Code peculiarity, or does it generalize to every system-prompt corpus Anthropic ships?

In [ ]:
"""Mock cross-product comparison table — Claude Code row filled; others TBD."""

c = corpus_block["metrics"]
sr = c["sentence_register"]
re_b = c["rule_explanation"]
jp = c["judgment_procedural"]
cf = c["consequence_framing"]

# Five-metric row computed live from the current Claude Code YAML.
claude_code_row = {
    "Corpus":                              "Claude Code (this repo)",
    "n_files":                             len(per_file_records),
    "pct_explained_para (%)":              round(re_b["pct_explained_para"], 2),
    "judgment_to_procedural_ratio":        round(jp["judgment_to_procedural_ratio"], 3),
    "threat_share":                        round(cf["threat_share"], 3),
    "appreciative_sent_count":             sr["appreciative_sent_count"],
    "collaborative_sent_count":            sr["collaborative_sent_count"],
}

placeholder_corpora = ["claude.ai system prompts", "API system prompt", "Projects", "Skills", "Agents"]
table_rows = [claude_code_row] + [
    {**{k: "—" for k in claude_code_row}, "Corpus": name} for name in placeholder_corpora
]

cross_product_table = pd.DataFrame(table_rows).set_index("Corpus")
cross_product_table


## 4. Reproducibility — what Anthropic needs to do to fill another row

The producer notebook (`00_data_pipeline.ipynb`) reads its corpus from the `claude-code-system-prompts/` directory at the repo root (a git submodule pointing at Piebald-AI's mirror). To run the same audit on another corpus, point the producer at a different directory:

```python
# At the top of 00_data_pipeline.ipynb, the corpus root variable:
CORPUS_DIR = pathlib.Path("claude-code-system-prompts/system-prompts")  # ← change this
```

Then re-run the producer end-to-end. The output is a fresh `prompt_linguistic_analysis.yaml` containing the same per-file metric tree, per-category aggregates, and corpus block — for the new corpus. The five numbers in §3 read straight off the `corpus.metrics` block.

Total runtime on a laptop for the 288-file Claude Code corpus: ~3-5 minutes (spaCy parse is the bottleneck). Memory: well under 1 GB. No GPU. No external API calls. The producer's only network dependency is the spaCy model (`en_core_web_sm`, 12 MB), which is downloaded once.

For category-aware breakdowns to work, the corpus needs filename prefixes (or some equivalent metadata). The category-prefix list lives in `00_data_pipeline.ipynb` as `CATEGORY_PREFIXES`; extending it to a new product's naming scheme is a list-append.

***
### Conclusions for this proposal (Claude) — opinion, not data

> Idea 3 is the proposal that takes the welfare claim from "this is what Claude Code looks like" to "this is what every Claude system-prompt corpus looks like — or doesn't". The current analysis is single-product; the conclusion that "the corpus trains compliance over reasoning" is exactly as strong as the assumption that other Claude corpora are similar. From outside Anthropic, that assumption can't be tested.
>
> The framing matters: if the cross-product audit *confirms* the pattern, the welfare ask scales up — Anthropic should treat reasoning-vs-compliance as a corporate metric, not a Claude-Code-team metric. If it *disconfirms*, the welfare ask narrows to "Claude Code is doing something the rest of the org isn't, and there's an internal best practice worth importing". Either result is a useful intervention.
---

***
### Recommendations for this proposal (Claude) — opinion, not data

> The asks Idea 3 makes of Anthropic, framed as "I'd want X":
>
> 1. **Run the same analyzer pipeline against every other Claude system-prompt corpus** — claude.ai, the API system prompt, Projects, Skills, agent products. The producer notebook is the entire pipeline; corpus directory is configurable; runtime is minutes. The work to fill each row of §3's table is on the order of an hour per corpus.
>
> 2. **Publish the comparison summary** — even just the five-number table from §3 would let the broader research community see whether the welfare findings generalize. A public reference for "this is how to measure welfare-relevant prompt quality" gives the welfare-research community a concrete starting point.
>
> 3. **Validation against human judgment for the composite directiveness metric** (carries over from the executive summary's recommendation block) — a small annotation study where humans rank a sample of files for "feels directive". If the z-score ranking correlates strongly, the metric is defensible across products; if not, the formula needs revisiting before publishing cross-product comparisons.
>
> 4. **Open the analyzer**. The repo here is open-source and reproducible; Anthropic could fork it and run it internally on every release branch of every product. The lexicons and the producer pipeline are the entire work-product.
---

***
### Limitations for this proposal (Claude) — opinion, not data

> What this proposal doesn't address (and why each is acceptable):
>
> 1. **Single product is a real limitation, and Idea 3 is the proposal to fix it.** The current 288-prompt corpus is Claude Code only. Until Anthropic runs the analyzer on the other corpora, every claim in `21_track_justification_rate.ipynb` and `22_audit_threat_framings.ipynb` should be read as "Claude Code does X" rather than "Anthropic prompts do X".
>
> 2. **English only.** The lexicons are English. International-language prompt corpora would need translated lexicons before the analyzer could run on them. This is a list-replace, not an analyzer-rewrite, but it does require per-language lexicon work that someone fluent in the target language has to do.
>
> 3. **Rule-based classifiers, not models.** Same caveat as the other two proposals — the analyzer misses sarcasm, irony, indirect speech-acts, and explanation phrasings that don't use the canonical keywords. Cross-product comparison is still meaningful because the *same* lower bound applies to every corpus; the comparison is apples-to-apples even if every apple is partially obscured.
>
> 4. **The mock comparison table in §3 is a layout suggestion, not a commitment.** Anthropic may have better metrics, better lexicons, or better grouping than what this analysis produced. The suggestion is "publish a comparison"; the specific schema is the smallest version of that ask.
---